In [11]:
import geopandas as gpd
import os
import osmnx as ox
import pandas as pd
import psycopg2
from shapely.geometry import LineString

In [12]:
host = "localhost"
database = "ukraine_nature_db"
user = os.getenv("SQL_USER")
password = os.getenv("SQL_PASSWORD")

In [13]:
conn = psycopg2.connect(
    host=host, database=database, user=user, password=password
)
cursor = conn.cursor()

In [ ]:
import requests
import numpy as np
import geopandas as gpd
from shapely.ops import linemerge, unary_union
from shapely.geometry import LineString

def save_route_to_db(gpx_file_path, route_name, route_type, difficulty, category, author, description):
    try:
        cursor.execute("SELECT setval(pg_get_serial_sequence('routes', 'route_id'), COALESCE(MAX(route_id), 0) + 1, false) FROM routes;")
        conn.commit()
    except Exception:
        conn.rollback()


    cursor.execute("SELECT COUNT(*) FROM routes WHERE name = %s;", (route_name,))
    exists = cursor.fetchone()[0]
    if exists > 0:
        print(f"⚠️ Route '{route_name}' already exists in the database. Skipping insertion.\n")
        return


    gdf_tracks = gpd.GeoDataFrame()
    priority_layers = ['tracks', 'routes', 'track_points', 'route_points']
    
    for layer in priority_layers:
        try:
            temp_gdf = gpd.read_file(gpx_file_path, layer=layer)
            temp_gdf = temp_gdf[temp_gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
            if not temp_gdf.empty:
                gdf_tracks = temp_gdf
                break
        except Exception:
            continue
            

    if gdf_tracks.empty:
        try:
            temp_gdf = gpd.read_file(gpx_file_path)
            gdf_tracks = temp_gdf[temp_gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
        except Exception:
            pass

    if gdf_tracks.empty:
        raise ValueError("No linear tracks or routes found in the file!")


    gdf_projected = gdf_tracks.to_crs(epsg=32635)
    

    combined_geom = unary_union(gdf_projected.geometry.tolist())
    if combined_geom.geom_type == 'MultiLineString':
        try:
            combined_geom = linemerge(combined_geom)
        except Exception:
            pass

    total_length_m = combined_geom.length
    dist_km = round(total_length_m / 1000, 2)


    has_z_in_track = False
    extracted_elevations = []

    for geom in gdf_tracks.geometry:
        if geom:
            lines = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
            for line in lines:
                coords = list(line.coords)
                if len(coords) > 0 and len(coords[0]) >= 3:
                    has_z_in_track = True
                    break
        if has_z_in_track:
            break


    if has_z_in_track:
        for geom in gdf_tracks.geometry:
            if geom:
                lines = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
                for line in lines:
                    for coord in line.coords:
                        if len(coord) >= 3:
                            extracted_elevations.append(coord[2])
        print("⛰ Elevations successfully extracted directly from GPX file.")
    else:
        print("🌐 No elevation data in track — fetching from Elevation API in chunks...")
        
        step_m = 500.0
        distances = np.arange(0, total_length_m, step_m)
        if len(distances) == 0 or distances[-1] != total_length_m:
            distances = np.append(distances, total_length_m)

        interpolated_points = [combined_geom.interpolate(d) for d in distances]
        pts_wgs84 = gpd.GeoSeries(interpolated_points, crs="EPSG:32635").to_crs("EPSG:4326")
        
        lats = [pt.y for pt in pts_wgs84]
        lons = [pt.x for pt in pts_wgs84]

        chunk_size = 100
        extracted_elevations = []
        
        for i in range(0, len(lats), chunk_size):
            chunk_lats = lats[i:i + chunk_size]
            chunk_lons = lons[i:i + chunk_size]
            
            url = f"https://api.open-meteo.com/v1/elevation?latitude={','.join(map(str, chunk_lats))}&longitude={','.join(map(str, chunk_lons))}"
            response = requests.get(url)
            
            if response.status_code == 200:
                elevs = response.json().get("elevation", [])
                extracted_elevations.extend(elevs)
            else:
                extracted_elevations.extend([0.0] * len(chunk_lats))


    total_gain = 0.0
    for i in range(1, len(extracted_elevations)):
        z_diff = extracted_elevations[i] - extracted_elevations[i-1]
        if z_diff > 1.5:
            total_gain += z_diff


    base_time = dist_km / 4.5
    elevation_time_penalty = (total_gain / 100) * 0.4
    raw_duration = base_time + elevation_time_penalty
    duration_hours = round(raw_duration * 2) / 2
    if duration_hours < 0.5:
        duration_hours = 0.5


    final_geom_wgs84 = gpd.GeoSeries([combined_geom], crs="EPSG:32635").to_crs("EPSG:4326").iloc[0]
    wkt_geom = final_geom_wgs84.wkt


    insert_query = """
        INSERT INTO routes (
            name, route_type, difficulty, category, author_info, 
            description, distance_km, elevation_gain_m, duration_hours, geom
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, ST_Multi(ST_GeomFromText(%s, 4326)));
    """

    cursor.execute(
        insert_query,
        (
            route_name,
            route_type,
            difficulty,
            category,
            author,
            description,
            int(dist_km),
            int(total_gain),
            float(duration_hours),
            wkt_geom
        )
    )
    
    conn.commit()
    print(f"✅ Route '{route_name}' successfully saved to database!")
    print(f"   📏 Distance: {dist_km} km | 📈 Elevation Gain: {int(total_gain)} m | ⏱ Duration: {duration_hours} hrs.\n")

In [31]:
save_route_to_db(
    gpx_file_path=r"D:\QGIS\project_UKRAINE\Camino Podolico.gpx",
    route_name="Camino Podolico",
    route_type="пішохідний",
    difficulty="важкий",
    category="спортивний",
    author="caminopodolico.net",
    description="Подільський шлях святого Якова. Маршрут на сьогодні поєднує 14 точок Вінницької та Хмельницької областей. Тривалість маршруту: 14-16 днів. Видається паспорт і сертифікат про проходження."
)

⚠️ Route 'Camino Podolico' already exists in the database. Skipping insertion.



In [32]:
save_route_to_db(
    gpx_file_path=r"D:\QGIS\project_UKRAINE\Lumshory Zhdeniievo.gpx",
    route_name="с. Лумшори — с-ще. Жденієво",
    route_type="пішохідний",
    difficulty="важкий",
    category="спортивний",
    author="karpaty.rocks",
    description="с. Лумшори через Полонину Руна, ур. Прелука, г. Остра Гора, с. Збини до с-ще. Жденієво. Шлях через Лумшорські водоспади. Тривалість маршруту: 2 дні."
)

⚠️ Route 'с. Лумшори — с-ще. Жденієво' already exists in the database. Skipping insertion.



In [34]:
save_route_to_db(
    gpx_file_path=r"D:\QGIS\project_UKRAINE\Bakota.gpx",
    route_name="с. Фурманівка — Бакота",
    route_type="пішохідний",
    difficulty="важкий",
    category="спортивний",
    author="Назар Назарук (gorgany.com)",
    description="с. Фурманівка → Суржинецький яр → Китайгород → Урочище Окунь → с. Демшин → с. Субіч → с. Калачківці → c. Колодіївка → Гора-останець 'Бабин писок' → Бакотська затока. Тривалість маршруту: 2-3 дні."
)

🌐 No elevation data in track — fetching from Elevation API in chunks...
✅ Route 'с. Фурманівка — Бакота' successfully saved to database!
   📏 Distance: 44.36 km | 📈 Elevation Gain: 891 m | ⏱ Duration: 13.5 hrs.

